# Faruq-v3 ACMC1 -- paired three-seed confirmation

Konfirmasi validation-only yang dipraregistrasi setelah ACMC1 seed 42 PASS. Untuk tiap seed (42, 123, 2026), ACMC1 dibandingkan terhadap checkpoint D0 dari seed yang sama. D0/ACMC yang lengkap akan dipakai ulang; hanya run yang belum lengkap yang dilatih. Test tidak ada dan tidak boleh dibuka.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
BASELINE_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-yolo26n-baseline-v1'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc-one-stage-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
for seed in (42, 123, 2026):
    d0 = BASELINE_ROOT / f'D0_seed{seed}/weights/best.pt'
    acmc = OUTPUT_ROOT / f'ACMC1_seed{seed}/weights/best.pt'
    print(f'SEED {seed}: D0=' + ('COMPLETE' if d0.is_file() else 'START') + ', ACMC1=' + ('COMPLETE' if acmc.is_file() else 'START'))
print('GPU:', torch.cuda.get_device_name(0))
print('BASELINE ROOT:', BASELINE_ROOT)
print('ACMC ROOT    :', OUTPUT_ROOT)

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_acmc_confirmation',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--baseline-root', str(BASELINE_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--seeds', '42', '123', '2026', '--device', '0', '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Konfirmasi ACMC1 gagal, return code={return_code}; traceback lengkap tercetak di atas.')

In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/acmc1_three_seed_confirmation.json'
assert SUMMARY.is_file(), f'Konfirmasi belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
rows = []
for metric, values in result['aggregate'].items():
    rows.append({'metric': metric, **values})
display(pd.DataFrame(rows).style.format({key: '{:.2%}' for key in ('d0_mean', 'acmc1_mean', 'delta_mean', 'delta_min')}))
print('CRITERIA:', result['criteria'])
print('DECISION:', result['decision'])
print('NEXT    :', result['next_action'])
print('SUMMARY :', SUMMARY)
print('Kirim tabel dan keputusan. Jangan membuka test secara manual.')